In [1]:
from tqdm.auto import tqdm
from ingest import load_faq_data
from sentence_transformers import SentenceTransformer

In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
documents = load_faq_data()
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

In [14]:
import psycopg

conn = psycopg.connect(
    # host="127.0.0.1",
    host="localhost",
    port=5432,
    dbname="faq",
    user="pgvector",
    password="password",
)

conn.execute("CREATE EXTENSION IF NOT EXISTS vector")

<psycopg.Cursor [COMMAND_OK] [INTRANS] (host=localhost user=pgvector database=faq) at 0x1df22532680>

In [15]:
conn.execute("""
    DROP TABLE IF EXISTS documents
""")

conn.execute("""
    CREATE TABLE documents (
        id SERIAL PRIMARY KEY,
        course TEXT,
        section TEXT,
        question TEXT,
        answer TEXT,
        embedding vector(384)
    )
""")

<psycopg.Cursor [COMMAND_OK] [INTRANS] (host=localhost user=pgvector database=faq) at 0x1df1f666680>

In [16]:
zip(documents,vectors)

In [17]:
def vec_to_str(vector):
    return "[" + ",".join(str(x) for x in vector) + "]"

for doc, vec in tqdm(zip(documents, vectors), total=len(documents)):
    conn.execute(
        """
        INSERT INTO documents (course, section, question, answer, embedding)
        VALUES (%s, %s, %s, %s, %s::vector)
        """,
        (doc["course"], doc["section"], doc["question"], doc["answer"],
         vec_to_str(vec))
    )

conn.commit()

  0%|          | 0/1350 [00:00<?, ?it/s]

In [18]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)
query_str = vec_to_str(query_vector)

In [19]:
query_str

'[-0.00947486,-0.06923238,-0.029059537,0.012938978,0.013622831,0.00023433844,-0.07741694,-0.0913697,-0.104661286,-0.019223649,-0.043045968,0.03964786,0.004325218,0.04924712,0.00818579,-0.041844994,-0.04338224,-0.0253527,-0.0013160974,-0.0017762207,-0.08884508,0.044900175,-0.026151262,0.023449602,-0.0091807265,0.013769341,0.018569155,0.08787835,-0.032130916,-0.07984384,-0.036902785,0.06971703,0.031200482,0.029062623,0.0049837586,0.017343337,0.06409656,-0.05677014,0.0065930365,0.022662386,-0.042738024,-0.027881954,-0.012338484,0.050004482,0.0309628,0.09940238,-0.059881907,-0.085765295,0.01954838,0.030833432,0.025987705,-0.046615645,-0.00039917819,0.011001665,-0.004584892,0.07484974,0.02326188,0.028910313,-0.112479344,-0.007625577,-0.010046852,-0.047283716,-0.07600153,0.054186583,0.01966646,0.018858792,-0.048078943,-0.014167341,0.12337664,-0.07372961,0.00057705294,-0.016402308,0.0370184,0.0066005685,0.09973395,0.016072491,0.068566605,-0.01510555,0.08021406,-0.03827427,-0.024316037,0.08188

In [20]:
results = conn.execute(
    """
    SELECT course, question, answer,
           1 - (embedding <=> %s::vector) AS similarity
    FROM documents
    ORDER BY embedding <=> %s::vector
    LIMIT 5
    """,
    (query_str, query_str)
).fetchall()

for row in results:
    print(f"[{row[0]}] {row[1]} (similarity: {row[3]:.4f})")

[llm-zoomcamp] I just discovered the course. Can I still join? (similarity: 0.8365)
[machine-learning-zoomcamp] The course has already started. Can I still join it? (similarity: 0.6904)
[mlops-zoomcamp] Course - Can I still join the course after the start date? (similarity: 0.6043)
[data-engineering-zoomcamp] Course: Can I still join the course after the start date? (similarity: 0.5959)
[data-engineering-zoomcamp] Course: Can I get support if I take the course in the self-paced mode? (similarity: 0.5927)


In [21]:
results = conn.execute(
    """
    SELECT course, question, answer,
           1 - (embedding <=> %s::vector) AS similarity
    FROM documents
    WHERE course = %s
    ORDER BY embedding <=> %s::vector
    LIMIT 5
    """,
    (query_str, "llm-zoomcamp", query_str)
).fetchall()

In [22]:
conn.execute("""
    CREATE INDEX ON documents
    USING hnsw (embedding vector_cosine_ops)
""")

<psycopg.Cursor [COMMAND_OK] [INTRANS] (host=localhost user=pgvector database=faq) at 0x1df225319c0>

In [23]:
def pgvector_search(query, course="llm-zoomcamp", num_results=5):
    query_vector = model.encode(query)
    query_str = vec_to_str(query_vector)
    rows = conn.execute(
        """
        SELECT course, section, question, answer
        FROM documents
        WHERE course = %s
        ORDER BY embedding <=> %s::vector
        LIMIT %s
        """,
        (course, query_str, num_results)
    ).fetchall()

    return [
        {"course": r[0], "section": r[1], "question": r[2], "answer": r[3]}
        for r in rows
    ]

In [24]:
results = pgvector_search("How do I join the course?")

In [25]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Can I run the course locally instead of Codespaces?',
  'answer': 'Yes. Codespaces is just the easiest way for everyone to start with the same environment.\n\nYou can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module.\n\nIf you run locally, make sure you document your setup and keep your environment reproducible.'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question':

In [ ]:
def vec_to_str(vector):
    return "[" + ",".join(str(x) for x in vector) + "]"

In [26]:
text = [0.12,98.1,23,]
vec_to_str(text)

'[0.12,98.1,23]'

In [27]:
for x in text:
    print(x)

0.12
98.1
23
